In [1]:
#rag_pipeline() Using TinyLLaMA

In [3]:
import pandas as pd
import faiss
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import numpy as np

# Load chunk texts
df = pd.read_csv(r"C:\Users\hp\Downloads\filtered_chunks.csv")
texts = df["chunk_text"].tolist()

# Load FAISS index
index = faiss.read_index(r"C:/Users/hp/Downloads/faiss_index1a.idx")

# Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load TinyLLaMA model + tokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
llama_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
).to("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
def retrieve_top_k_chunks(query, k=5):
    query_vector = embed_model.encode([query]).astype("float32")
    D, I = index.search(query_vector, k)
    return [texts[i] for i in I[0]]


In [5]:
def build_prompt(query, chunks):
    mapped = "\n\n---\n\n".join([f"{chunk}\n\nQuestion: {query}" for chunk in chunks])
    final_prompt = (
        "Answer the question below using the provided context. "
        "Think step by step and explain your reasoning.\n\n" + mapped
    )
    return final_prompt


In [6]:
def call_tiny_llama(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(llama_model.device)
    outputs = llama_model.generate(**inputs, max_new_tokens=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [7]:
def rag_pipeline(query):
    chunks = retrieve_top_k_chunks(query)
    prompt = build_prompt(query, chunks)
    answer = call_tiny_llama(prompt)
    return answer


In [8]:
query = "Who is marieve herington?"
answer = rag_pipeline(query)

print("\n Final Answer:")
print(answer)


This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.



 Final Answer:
Answer the question below using the provided context. Think step by step and explain your reasoning.

and members of the royal jelly orchestra, appearing on two more albums. she sang the theme songs for cbc television s sesame park, nelvana s pippi longstocking and tvontario s marigold s mathemagics. herington moved to los angeles in 2008. 2i the marieve herington band performs in southern california and toronto. their latest album is midnight sessions. on april 14, 2021, herington announced that she became an american citizen. filmography edit animation blossoming timely manor, 2006 e midnight sessions 2010 references 4,44 bio marieve herington. retrieved june 2, 2021. 44 kurek, dominik june 13, 2013. marieve herington gets her start on charlie s last season. metroland media group. retrieved october 8, 2013. by aabcd october 8, 2013. 4,44 marieve herington. retrieved october 8, 2013. jazz musician returns home 2013. marieve herington biography. archived from the origin

In [ ]:
#LLM Fallback Strategy

In [9]:
def call_llm(prompt):
    try:
        return call_openai(prompt)
    except Exception as e:
        print(" OpenAI failed. Falling back to TinyLLaMA.")
        return call_tiny_llama(prompt)


In [10]:
query = "Who is marieve herington?"
prompt = f"Answer the question clearly and step by step:\n\n{query}"
print(call_tiny_llama(prompt))


Answer the question clearly and step by step:

Who is marieve herington?

Answer: Marieve Herington is a British actress who has appeared in numerous films and TV shows, including "The Crown," "The Durrells," and "The Night Manager."


In [16]:
!pip install diskcache


Defaulting to user installation because normal site-packages is not writeable
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)


In [11]:
from diskcache import Cache

cache = Cache("./llm_cache")

def cached_response(query):
    if query in cache:
        return cache[query]
    else:
        chunks = retrieve_top_k_chunks(query)
        prompt = build_prompt(query, chunks)
        answer = call_llm(prompt)
        cache[query] = answer
        return answer


In [12]:
from diskcache import Cache

# Create or load a cache folder
cache = Cache("./llm_cache")

# Example usage
query = "Who is marieve herington? ?"
if query in cache:
    answer = cache[query]
else:
    answer = rag_pipeline(query)
    cache[query] = answer

print(answer)


Answer the question below using the provided context. Think step by step and explain your reasoning.

and members of the royal jelly orchestra, appearing on two more albums. she sang the theme songs for cbc television s sesame park, nelvana s pippi longstocking and tvontario s marigold s mathemagics. herington moved to los angeles in 2008. 2i the marieve herington band performs in southern california and toronto. their latest album is midnight sessions. on april 14, 2021, herington announced that she became an american citizen. filmography edit animation blossoming timely manor, 2006 e midnight sessions 2010 references 4,44 bio marieve herington. retrieved june 2, 2021. 44 kurek, dominik june 13, 2013. marieve herington gets her start on charlie s last season. metroland media group. retrieved october 8, 2013. by aabcd october 8, 2013. 4,44 marieve herington. retrieved october 8, 2013. jazz musician returns home 2013. marieve herington biography. archived from the original user generate